# RoPE from first principles

Rotary Position Embedding rotates each query and key by an angle
proportional to its absolute position, and the dot product between them
then depends only on their relative distance. Everything else is a
consequence of that one sentence.

This notebook derives the method rather than summarizing it. Every claim
is paired with code that runs.

**Assumed:** dot products, matrix multiplication, the shape of attention.
No familiarity with positional encoding schemes is needed.

**Covered:** why order is invisible to attention, why rotation is the
natural fix, why one frequency is not enough, why two valid conventions
can silently disagree, and why a wrong position never raises an error.

Reference: Su et al., *RoFormer: Enhanced Transformer with Rotary
Position Embedding*, [arXiv:2104.09864](https://arxiv.org/abs/2104.09864).


In [ ]:
import os
import subprocess
import sys
import inspect

import numpy as np
from IPython.display import Code, Image

sys.path.insert(0, os.path.abspath(".."))
ASSETS = os.path.join("..", "assets")

REPO_ROOT = os.path.abspath("..")

def show_source(obj):
    """Render a function's source as highlighted code."""
    return Code(inspect.getsource(obj), language="python")

def run_test(name):
    """Run a test script from the repo root and print its output."""
    result = subprocess.run(
        [sys.executable, os.path.join(REPO_ROOT, name)],
        capture_output=True, text=True, cwd=REPO_ROOT,
    )
    print(result.stdout, end="")
    if result.returncode != 0:
        print(result.stderr, end="")
    return result.returncode

---

## 1. Attention cannot see order

An attention score is a dot product:

$$
\text{score}(q_m, k_n) = \frac{q_m \cdot k_n}{\sqrt{d}}
$$

The dot product reads vector content. It has no access to a sequence
index. Reorder the tokens of a sentence without changing the tokens
themselves and every pairwise score is unchanged.

Chinese makes the point cleanly, because word order carries the meaning
with no inflection to help:

| Sentence | Meaning |
|---|---|
| 猫 吃 鱼 | the cat eats the fish |
| 鱼 吃 猫 | the fish eats the cat |

Same three tokens. Opposite claims about the world. Below, the score
between 猫 and 鱼 is computed in both sentences.

In [ ]:
tokens = {
    "猫": np.array([1.0, 0.3]),   # cat
    "吃": np.array([0.2, 0.9]),   # eats
    "鱼": np.array([0.8, -0.4]),  # fish
}

def pair_score(a, b):
    return float(np.dot(tokens[a], tokens[b]) / np.sqrt(2))

for sentence in (["猫", "吃", "鱼"], ["鱼", "吃", "猫"]):
    order = " ".join(sentence)
    print(f"{order}   score(猫, 鱼) = {pair_score('猫', '鱼'):.6f}")

The two scores are identical, so the mechanism is blind to order by
construction rather than by oversight. Position has to be injected from
outside.

The obvious move is additive: attach a position vector before the dot
product, $x + p$. Expanding $(x_a + p_m) \cdot (x_b + p_n)$ produces four
terms, three of which mix position with content, and nothing separates
them again. The network is left to learn a separation that was never
guaranteed to exist.

State the requirement instead, and look for a function that meets it:

$$
\langle f_q(x_m, m),\ f_k(x_n, n) \rangle = g(x_m,\, x_n,\, m - n)
$$

Read the two sides against each other. On the left, $f_q$ sees only $m$
and $f_k$ sees only $n$, because attention transforms them
independently. On the right, the result may depend on $m - n$ alone.

Two local operations, each knowing only its own absolute position, must
combine into something that knows only the difference.

---

## 2. Rotation satisfies the requirement

The dot product of two vectors in a plane is:

$$
a \cdot b = |a|\,|b| \cos(\phi_b - \phi_a)
$$

No absolute angle appears on the right side. Only the difference
$\phi_b - \phi_a$ does. This is a property of the dot product itself,
independent of attention or of any encoding scheme.

Rotate $a$ by $t_1$ and $b$ by $t_2$, independently:

$$
\phi_{b'} - \phi_{a'} = (\phi_b + t_2) - (\phi_a + t_1) = (\phi_b - \phi_a) + (t_2 - t_1)
$$

Set $t_1 = m\theta$ and $t_2 = n\theta$. The angle between the rotated
vectors differs from the original by exactly $(n - m)\theta$. The
requirement from section 1 is now met structurally, with no learning and
no approximation.

In the plane, treating a vector as a complex number turns rotation into
multiplication by a unit exponential:

$$
f_q(x_m, m) = (W_q x_m)\, e^{i m \theta}, \qquad
f_k(x_n, n) = (W_k x_n)\, e^{i n \theta}
$$

The inner product then yields:

$$
\mathrm{Re}\left[(W_q x_m)(W_k x_n)^{*}\, e^{i (m-n)\theta}\right]
$$

The term $e^{i(m-n)\theta}$ is not inserted. It falls out of
$e^{ia}e^{-ib} = e^{i(a-b)}$.

In [ ]:
from rope import rotation_matrix_2d  # dependency of rotate_2d below

def rotate_2d(v: np.ndarray, position: int, theta: float) -> np.ndarray:
    """Rotate a 2D vector by (position * theta) radians."""
    return rotation_matrix_2d(position * theta) @ v

# Guard against drift: fails loudly if rope.py changes and this cell does not.
import rope as _rope_module, inspect as _inspect
assert _inspect.getsource(_rope_module.rotate_2d) == _inspect.getsource(rotate_2d), \
    "rotate_2d above is out of sync with rope.py"

One property of the rotation matrix deserves attention now, because
section 7 depends on it:

$$
R^{\mathsf{T}} R = I, \qquad \det R = 1, \qquad \|Rv\| = \|v\|
$$

These are not conveniences selected for the method. They are what
rotation means in a Euclidean space. Norm is preserved for any angle,
whether that angle is correct or not.

`test_invariance.py` checks the two structural claims and the norm
property, including at negative positions that no correct
implementation would ever produce.

In [ ]:
run_test("test_invariance.py")

A vector rotated at increasing positions, with the coordinates that
would be stored in the Q or K tensor at each step. Position $m$ is not
reached by passing through $m-1$: each angle is computed directly.

In [ ]:
Image(filename=os.path.join(ASSETS, "equal_distance_animation.gif"))

---

## 3. The claim is stronger than it first appears

Shifting two positions by the same offset and finding the angle
unchanged is the weak reading. It is true of any rigid transform and
proves little.

The real claim is that any two position pairs sharing a difference
produce the same angle, even when the pairs are otherwise unrelated. It
follows from the composition property of rotations:

$$
R_m^{\mathsf{T}} R_n = R_{n-m}
$$

so that

$$
(R_m q_m)^{\mathsf{T}} (R_n k_n) = q_m^{\mathsf{T}} R_{n-m} k_n
$$

The pair $(m, n) = (7, 11)$ and the pair $(m, n) = (131, 135)$ share
nothing but their difference. The score is the same. This is checked by
`equal_difference_equal_score` in the test above, across position pairs
spanning several orders of magnitude.

The weak version first, for contrast: one fixed pair, shifted by a
growing offset.

In [ ]:
# [PLACEHOLDER] display assets/translation_invariance_animation.gif

Then the strong version: five unrelated pairs, sharing only their
difference.

In [ ]:
Image(filename=os.path.join(ASSETS, "equal_distance_animation.gif"))

---

## 4. One frequency is not enough

Query and key vectors have head dimension $d$, far larger than 2. Split
the space into $d/2$ independent planes and rotate each one. The
resulting matrix is block diagonal, each block a plane rotation:

$$
R_m = \begin{pmatrix}
\cos m\theta_1 & -\sin m\theta_1 & & & \\
\sin m\theta_1 & \cos m\theta_1 & & & \\
& & \ddots & & \\
& & & \cos m\theta_{d/2} & -\sin m\theta_{d/2} \\
& & & \sin m\theta_{d/2} & \cos m\theta_{d/2}
\end{pmatrix}
$$

The inner product is linear, so it decomposes across orthogonal
subspaces, and the argument of section 2 applies within each plane.

Each plane needs a frequency:

$$
\theta_i = \text{base}^{-2i/d}, \qquad i = 0, \dots, d/2 - 1
$$

The question worth asking is why $\theta_i$ varies with $i$ at all. A
single frequency would be simpler.

Cosine is periodic. With one frequency, $m\theta$ wraps past $2\pi$ and
distinct positions map onto nearly identical angles. For short sequences
this is invisible. At the lengths modern models handle, position 5 and
position 50005 become indistinguishable from their rotation alone. The
encoding aliases.

A spectrum solves this the way a clock does. A second hand alone cannot
express an hour. An hour hand alone cannot express a second. Together
the reading is unambiguous over a range neither could cover. Fast planes
($i$ small, $\theta_i$ near 1) separate nearby positions. Slow planes
($i$ large, $\theta_i$ near $1/\text{base}$) barely move across the whole
sequence and separate distant ones.

In [ ]:
def inverse_frequencies(dim: int, base: float = 10000.0) -> np.ndarray:
    """Return the RoPE frequency spectrum theta_i = base^(-2i/dim) for i in [0, dim/2).

    dim must be even. The result has shape (dim // 2,).
    """
    if dim % 2 != 0:
        raise ValueError(f"dim must be even, got {dim}")
    i = np.arange(dim // 2)
    return base ** (-2.0 * i / dim)

# Guard against drift: fails loudly if rope.py changes and this cell does not.
import rope as _rope_module, inspect as _inspect
assert _inspect.getsource(_rope_module.inverse_frequencies) == _inspect.getsource(inverse_frequencies), \
    "inverse_frequencies above is out of sync with rope.py"
for dim in (32, 128):
    freqs = inverse_frequencies(dim)
    print(f"dim={dim:3d}  planes={len(freqs):3d}  "
          f"theta_0={freqs[0]:.4f}  theta_last={freqs[-1]:.3e}")

Two cautions.

The number of planes is $d/2$ and nothing else. A head dimension of 128
gives 64 planes. Any figure showing 16 is showing a chosen example, not
a canonical count.

The value of `base` is empirical. It works, and section 8 covers a
second property it happens to produce, but it is not derived from first
principles. Read the actual value from the model config rather than
assuming 10000, since model families differ.

Materializing $R_m$ and calling a dense matmul wastes memory and
arithmetic on a matrix that is almost entirely zeros. The equivalent
elementwise form operates on pairs directly:

$$
\begin{aligned}
x'_{2i} &= x_{2i}\cos(m\theta_i) - x_{2i+1}\sin(m\theta_i) \\
x'_{2i+1} &= x_{2i}\sin(m\theta_i) + x_{2i+1}\cos(m\theta_i)
\end{aligned}
$$

Since the same $\cos$ and $\sin$ values apply at every position across
every head, they are computed once up to the maximum sequence length and
reused. In practice RoPE is a table lookup and two multiplies.

In [ ]:
def apply_rope_adjacent(x: np.ndarray, position: int, freqs: np.ndarray) -> np.ndarray:
    """Apply RoPE using adjacent-pair convention: dims (2i, 2i+1) rotate by position * freqs[i].

    This is the layout written in the RoFormer paper (Su et al. 2023, Eq. 15).
    x has shape (dim,); freqs has shape (dim // 2,).
    """
    dim = x.shape[-1]
    out = np.empty_like(x)
    angles = position * freqs
    cos, sin = np.cos(angles), np.sin(angles)
    x_even, x_odd = x[0::2], x[1::2]
    out[0::2] = x_even * cos - x_odd * sin
    out[1::2] = x_even * sin + x_odd * cos
    return out

# Guard against drift: fails loudly if rope.py changes and this cell does not.
import rope as _rope_module, inspect as _inspect
assert _inspect.getsource(_rope_module.apply_rope_adjacent) == _inspect.getsource(apply_rope_adjacent), \
    "apply_rope_adjacent above is out of sync with rope.py"

Sixteen planes from the spectrum, with a marker sweeping through
position $m$. At any single frame, the marker heights taken together are
that position's signature.

In [ ]:
Image(filename=os.path.join(ASSETS, "frequency_spectrum_animation.gif"))

---

## 5. Two conventions, one of them silent

The paper pairs adjacent dimensions: plane $i$ spans $(2i,\ 2i+1)$.

Common implementations pair split halves: plane $i$ spans
$(i,\ i + d/2)$. The cosine and sine tables are duplicated rather than
interleaved to match.

Both are valid rotary encodings. They are equivalent under a permutation
of dimension indices:

$$
\pi(i) = 2i, \qquad \pi(i + d/2) = 2i + 1, \qquad i = 0, \dots, d/2 - 1
$$

read as: the value at slot $\pi(j)$ in the adjacent layout belongs at
slot $j$ in the split-half layout.

In [ ]:
def adjacent_to_half_permutation(dim: int) -> np.ndarray:
    """Return the index permutation mapping adjacent-pair layout to split-half layout.

    perm[j] is the index in the adjacent layout whose value belongs at slot j
    in the split-half layout. Given x in adjacent layout, x[perm] is the
    equivalent input for apply_rope_half. Use np.argsort(perm) to map
    outputs back.
    """
    half = dim // 2
    perm = np.empty(dim, dtype=int)
    perm[:half] = np.arange(half) * 2        # first members of each pair
    perm[half:] = np.arange(half) * 2 + 1    # second members of each pair
    return perm

# Guard against drift: fails loudly if rope.py changes and this cell does not.
import rope as _rope_module, inspect as _inspect
assert _inspect.getsource(_rope_module.adjacent_to_half_permutation) == _inspect.getsource(adjacent_to_half_permutation), \
    "adjacent_to_half_permutation above is out of sync with rope.py"

The choice is arbitrary in isolation. It stops being arbitrary the
moment tensors cross a boundary, because trained weights encode the
convention they were trained under. Apply the other convention without
the permutation and section 2's orthogonality takes over: valid shapes,
plausible magnitudes, wrong semantics, no error raised.

The third test below asserts exactly that. The corruption is real and is
detectable only by comparison against a known-correct result.

In [ ]:
run_test("test_conventions.py")

In [ ]:
# [PLACEHOLDER] diagram: which dimension index pairs with which,
# under each convention

---

## 6. The score matrix

Attention scores form a matrix, one entry per query and key pair:

$$
\text{scores} = \frac{Q K^{\mathsf{T}}}{\sqrt{d}}
$$

Causal masking means query $t$ attends only to keys $0 \dots t$, so only
the lower triangle is ever computed. A softmax over each row turns the
scores into weights, which is the last step before the weighted sum over
values and is otherwise not relevant here.

The example vectors below are two dimensional so that every entry can be
checked by hand.

In [ ]:
q_vectors = np.array([[0.0, 0.5],
                      [0.9, 0.4],
                      [1.0, 0.0],
                      [1.0, 3.0]])
k_vectors = np.array([[0.3, 0.7],
                      [1.0, 1.0],
                      [1.0, 1.0],
                      [0.6, 1.0]])

scores = q_vectors @ k_vectors.T
mask = np.tril(np.ones_like(scores, dtype=bool))
print(np.where(mask, np.round(scores, 2), np.nan))

One dot product per frame, filled in causal order.

In [ ]:
Image(filename=os.path.join(ASSETS, "score_matrix_animation.gif"))

These scores carry no positional information: they are the state of
affairs from section 1, before rotation is applied.

In [ ]:
# [PLACEHOLDER] side by side: the same score matrix without RoPE and
# with RoPE applied, showing the scores change

---

## 7. A wrong position never raises an error

RoPE is applied to keys before they enter a KV cache. Cached keys are
post-rotation, and the positional frame they were computed in is baked
into their values. Slicing a tensor moves numbers. It does not unwind a
rotation that already happened.

Consider a cache of length $p + s$ truncated down to its last $s$
entries. The surviving keys still hold rotations from absolute positions
$[p,\ p+s)$. If the next query's position is then taken from the
truncated cache length, it lands at $s$ while the keys sit near $p + s$:

$$
m - n \in (-p,\ s - p]
$$

Once $p$ exceeds $s$, every relative distance is negative. The query is
rotated as though the keys were ahead of it, an arrangement causal
training never produces.

Now recall $\|Rv\| = \|v\|$ from section 2. Nothing breaks. No
exception, no NaN, no shape mismatch, no warning. The vectors have the
lengths they always had. Only direction moved, and direction is
precisely what the dot product reads. The error lands in the quantity
that decides attention, in the one dimension of the representation that
no invariant is guarding.

A system that fails loudly teaches you where it failed. This one does
not. Correctness here cannot be inferred from the absence of errors.

In [ ]:
run_test("test_position_mismatch.py")

Divergence is present at every length tested. The magnitude of that
divergence does not follow a monotonic trend with sequence length in
this synthetic setting, and no such trend is claimed. Section 9 covers
why.

In [ ]:
Image(filename=os.path.join(ASSETS, "attention_mismatch_animation.gif"))

In [ ]:
# [PLACEHOLDER] third panel: amplified difference between the correct
# and mismatched attention weights

---

## 8. Decay comes for free

Group the entries of $q$ and $k$ into pairs and write the rotary inner
product as a sum of complex terms. Applying an Abel transformation to
that sum bounds it by:

$$
\left|\sum_{i=0}^{d/2-1} h_i\, e^{i(m-n)\theta_i}\right|
\le \max_i |h_{i+1} - h_i| \sum_{j=0}^{d/2-1} |S_j|,
\qquad S_j = \sum_{i=0}^{j-1} e^{i(m-n)\theta_i}
$$

The quantity that decays is the average of $|S_j|$ over $j$. It depends
only on the relative distance and the frequency spectrum, not on any
particular $q$ and $k$.

In [ ]:
def relative_upper_bound(distance, freqs):
    """Mean of |S_j| over partial sums of exp(1j * distance * theta_i)."""
    partial_sums = np.cumsum(np.exp(1j * distance * freqs))
    return float(np.mean(np.abs(partial_sums)))

freqs = inverse_frequencies(64)
for d in (1, 10, 50, 100, 250):
    print(f"distance {d:4d}   bound {relative_upper_bound(d, freqs):6.2f}")

The trend is downward but not monotonic. The bound is a sum of
oscillating terms, so individual distances can sit above their
neighbours while the envelope still falls. Reading a single pair of
distances proves nothing here; the shape over a range is the claim.

This closes the loop with section 4. The spectrum was chosen to avoid
aliasing. Decay arrives with it, unrequested. One design decision, two
useful consequences, which is a reasonable signal that a construction is
not arbitrary.

In [ ]:
Image(filename=os.path.join(ASSETS, "long_term_decay_animation.gif"))

---

## 9. Scope

Everything above concerns the encoding as a mathematical object. None of
it describes how a trained model responds.

The distinction is easy to lose and worth holding. Rotation does not
know what a sentence is. A statement like "the error is worse on short
sequences" is a statement about learned weights and a training
distribution, and no amount of testing against random vectors can
establish it. Such a claim needs measurement against the model in
question.

What the tests here do establish: the invariance holds, the norm
survives, the two conventions agree under permutation and disagree
without it, and a position inconsistent with cached rotations moves the
attention output. Those are properties of the encoding, and they hold
regardless of what any model does with them.

For per-function notes and derivations, see `rope_implementation_docs.md`.
For the derivation without code, see `RoPE_principle.md`.

Su et al., *RoFormer: Enhanced Transformer with Rotary Position
Embedding*, [arXiv:2104.09864](https://arxiv.org/abs/2104.09864).